# COE 311K Midterm Project: U.S. GDP Growth Rate Analysis
**Date**: March 2026

## Project Overview
This notebook applies curve-fitting and interpolation techniques to analyze the U.S. quarterly GDP growth rates from 2010 to 2023. We compare the exact interpolation method (Natural Cubic Splines) with approximation models (Polynomial Least Squares and Linear Regression). 

All algorithms—including solving tridiagonal systems and normal equations—are implemented entirely from scratch without using pre-built curve-fitting libraries.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

# We load the full GDP dataset to have the 56 quarters (2010 Q1 through 2023 Q4)
# We will identify the subset points mentioned in the prompt by exact matching or by index mapping.
# The prompt provides indices x = 1, 2, ..., n mapped to quarterly data. 
# 2010 Q1 is x = 1.
# 2023 Q4 is x = 56.

dates = []
full_y = []
subset_quarters = [
    ('2010', 'Q1'), ('2011', 'Q1'), ('2012', 'Q1'), ('2013', 'Q1'),
    ('2014', 'Q1'), ('2014', 'Q3'), ('2015', 'Q2'), ('2016', 'Q1'),
    ('2016', 'Q3'), ('2016', 'Q4'), ('2017', 'Q1'), ('2018', 'Q1'),
    ('2019', 'Q1'), ('2020', 'Q1'), ('2020', 'Q2'), ('2020', 'Q3'),
    ('2021', 'Q1'), ('2022', 'Q1'), ('2023', 'Q2'), ('2023', 'Q4')
]

with open('gdp_data.csv', 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        dates.append(row[0])
        full_y.append(float(row[1]))

start_idx = dates.index('2010-01-01')
end_idx = dates.index('2023-10-01')

x_full = np.arange(1, 57)
y_full = np.array(full_y[start_idx:end_idx+1])
date_full = dates[start_idx:end_idx+1]

# Reconstruct subset logically
month_map = {'Q1': '01-01', 'Q2': '04-01', 'Q3': '07-01', 'Q4': '10-01'}
x_subset = []
y_subset = []

for year, q in subset_quarters:
    date_str = f"{year}-{month_map[q]}"
    idx = date_full.index(date_str)
    x_subset.append(idx + 1)
    y_subset.append(y_full[idx])

x_subset = np.array(x_subset, dtype=float)
y_subset = np.array(y_subset, dtype=float)

print(f"Full dataset size: {len(x_full)}")
print(f"Subset size: {len(x_subset)}")



## Part A — Cubic Spline Interpolation

**1. Natural Cubic Spline Setup**

A cubic spline $S(x)$ consists of piecewise cubic polynomials $S_i(x)$ on each interval $[x_i, x_{i+1}]$ for $i = 1, \dots, n-1$.

Let $S_i(x) = a_i + b_i(x-x_i) + c_i(x-x_i)^2 + d_i(x-x_i)^3$.

**Unknowns**: For $n$ points, there are $n-1$ intervals, yielding $4(n-1)$ unknowns ($a_i, b_i, c_i, d_i$).
**Continuity Conditions**:
- $C^0$: $S_i(x_i) = y_i$ and $S_i(x_{i+1}) = y_{i+1}$ (Interpolation and $C^0$ continuity)
- $C^1$: $S_i'(x_{i+1}) = S_{i+1}'(x_{i+1})$ (First derivative continuity)
- $C^2$: $S_i''(x_{i+1}) = S_{i+1}''(x_{i+1})$ (Second derivative continuity)

Substituting $c_i = S''(x_i)/2$ gives a tridiagonal system for the variables $c_i$:
$h_{i-1} c_{i-1} + 2(h_{i-1} + h_i) c_i + h_i c_{i+1} = \frac{3}{h_i}(y_{i+1} - y_i) - \frac{3}{h_{i-1}}(y_i - y_{i-1})$
where $h_i = x_{i+1} - x_i$. This results in $n-2$ equations.

**Boundary Conditions**: For a "Natural" spline, the second derivative at the endpoints is zero, so $c_1 = 0$ and $c_n = 0$. This closes the system, allowing us to solve for the $n-2$ interior $c_i$ values efficiently using a tridiagonal solver, reducing a $4n 	imes 4n$ system to $(n-2) 	imes (n-2)$.


In [ ]:
def thomas_algorithm(a, b, c, d):
    """
    Solves a tridiagonal matrix system using the Thomas Algorithm.
    a: lower diagonal (length n-1)
    b: main diagonal (length n)
    c: upper diagonal (length n-1)
    d: right-hand side (length n)
    """
    n = len(d)
    c_prime = np.zeros(n-1)
    d_prime = np.zeros(n)
    x = np.zeros(n)
    
    c_prime[0] = c[0] / b[0]
    d_prime[0] = d[0] / b[0]
    
    for i in range(1, n-1):
        m = b[i] - a[i-1] * c_prime[i-1]
        c_prime[i] = c[i] / m
        d_prime[i] = (d[i] - a[i-1] * d_prime[i-1]) / m
        
    m = b[n-1] - a[n-2] * c_prime[n-2]
    d_prime[n-1] = (d[n-1] - a[n-2] * d_prime[n-2]) / m
    
    x[n-1] = d_prime[n-1]
    for i in range(n-2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i+1]
        
    return x

def natural_cubic_spline(x, y):
    n = len(x)
    h = np.diff(x)
    
    # Setup tridiagonal system for c (second derivatives)
    num_eqs = n - 2
    A_lower = h[1:num_eqs]
    A_main = 2 * (h[:-1] + h[1:])
    A_upper = h[1:num_eqs]
    
    RHS = np.zeros(num_eqs)
    for i in range(1, n-1):
        RHS[i-1] = (3 / h[i]) * (y[i+1] - y[i]) - (3 / h[i-1]) * (y[i] - y[i-1])
        
    # Solve system using Thomas Algorithm
    c_interior = thomas_algorithm(A_lower, A_main, A_upper, RHS)
    
    # Include boundary conditions c_1 = 0, c_n = 0
    c = np.zeros(n)
    c[1:-1] = c_interior
    
    # Compute a, b, d coefficients
    a = y[:-1]
    b = np.zeros(n-1)
    d = np.zeros(n-1)
    for i in range(n-1):
        b[i] = (y[i+1] - y[i]) / h[i] - h[i] * (2*c[i] + c[i+1]) / 3
        d[i] = (c[i+1] - c[i]) / (3 * h[i])
        
    return a, b, c[:-1], d

def evaluate_spline(x_eval, x_knots, coeffs):
    a, b, c, d = coeffs
    y_eval = np.zeros_like(x_eval)
    for idx, xi in enumerate(x_eval):
        # Handle evaluation points exactly on or outside boundaries
        if xi <= x_knots[0]:
            i = 0
        elif xi >= x_knots[-1]:
            i = len(x_knots) - 2
        else:
            i = np.where(x_knots <= xi)[0][-1]
            if i == len(x_knots) - 1:
                i -= 1 # Keep it in bounds
        
        dx = xi - x_knots[i]
        y_eval[idx] = a[i] + b[i]*dx + c[i]*dx**2 + d[i]*dx**3
    return y_eval

# Compute spline coefficients for our subset data
coeffs = natural_cubic_spline(x_subset, y_subset)

# 2. Evaluate at every full dataset quarter interval
y_spline_eval = evaluate_spline(x_full, x_subset, coeffs)

plt.figure(figsize=(10, 6))
plt.plot(x_full, y_full, 'k-', alpha=0.3, label='Actual Data (Not strictly given, but plotted for comparison)')
plt.scatter(x_subset, y_subset, color='red', label='Constraint Subset Points', zorder=5)
plt.plot(x_full, y_spline_eval, 'b-', label='Natural Cubic Spline')
plt.title("U.S. GDP Growth: Natural Cubic Spline Interpolation")
plt.xlabel("Quarters since 2010 Q1")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()



**3. Discussion: Outliers and the Runge Phenomenon**

While the spline interpolant perfectly connects all the specified data points, it exhibits severe oscillations (the Runge phenomenon). Our interpolating spline must stretch dramatically to precisely touch the massive COVID-19 GDP plunge at $t=42$ (-28.0%) and the resulting recovery spike at $t=43$ (34.9%). 

Because splines force $C^2$ continuity (the curve's concavity must match smoothly across the points), dragging an interpolant down so aggressively and then shooting it instantly back up causes large ripples/overshoots in the preceding and succeeding segments. 

**Alternative Approaches**:
For extreme outliers like the pandemic shock, exact interpolation ensures visual instability because of high first and second derivatives imposed on neighboring points. A **smoothing spline** (which penalizes extreme curvature) or a **weighted least squares** model (which could down-weight the COVID impact) would be infinitely preferable to capture general economic behaviors while allowing a minor error at the extreme shock points.


## Part B — Polynomial & Least Squares Comparison

**1. Degree-4 Polynomial Fit**

To find the polynomial $P_4(x) = p_0 + p_1x + p_2x^2 + p_3x^3 + p_4x^4$ that best fits the data, we establish the Vandermonde matrix $A$ where $A_{ij} = x_i^j$. Our overdetermined system is $A p pprox y$. 
We solve this using the Normal Equations: $A^T A p = A^T y$. Let's examine the condition number to avoid numerical instability.



In [ ]:
# 1. Degree-4 Polynomial Setup
degree = 4
A_poly = np.vander(x_subset, degree + 1, increasing=True)
cond_A = np.linalg.cond(A_poly.T @ A_poly)
print(f"Condition number of A^T A (raw x): {cond_A:.2e}")

# Given cond_A is very large, normalizing X helps numerical stability
x_subset_norm = (x_subset - np.mean(x_subset)) / np.std(x_subset)
A_poly_norm = np.vander(x_subset_norm, degree + 1, increasing=True)
cond_A_norm = np.linalg.cond(A_poly_norm.T @ A_poly_norm)
print(f"Condition number of A^T A (normalized x): {cond_A_norm:.2e}")

# Solve Normal Equations
b_poly = A_poly_norm.T @ y_subset
p_coeffs = np.linalg.solve(A_poly_norm.T @ A_poly_norm, b_poly)

# Evaluate and Plot
x_full_norm = (x_full - np.mean(x_subset)) / np.std(x_subset)
y_poly_eval = evaluate_polynomial(x_full_norm, p_coeffs)

def evaluate_polynomial(x_vals, p_coeffs):
    y_vals = np.zeros_like(x_vals)
    for j, c in enumerate(p_coeffs):
        y_vals += c * (x_vals**j)
    return y_vals

y_poly_eval = evaluate_polynomial(x_full_norm, p_coeffs)

plt.figure(figsize=(10, 6))
plt.scatter(x_subset, y_subset, color='red', label='Subset Points', zorder=5)
plt.plot(x_full, y_spline_eval, 'b--', alpha=0.5, label='Cubic Spline (Interpolation)')
plt.plot(x_full, y_poly_eval, 'g-', linewidth=2, label='Degree-4 Least Squares (Approximation)')
plt.title("Spline Interpolation vs. Degree-4 Polynomial Approximation")
plt.xlabel("Quarters since 2010 Q1")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()

# Residual plot for Polynomial Fit
y_poly_fit_subset = evaluate_polynomial(x_subset_norm, p_coeffs)
residuals = y_subset - y_poly_fit_subset

plt.figure(figsize=(10, 4))
plt.scatter(x_subset, residuals, color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.title("Residuals: Degree-4 Polynomial Fit")
plt.xlabel("Quarters")
plt.ylabel("Residual (Actual - Fitted)")
plt.grid(True)
plt.show()



**Approximation vs Interpolation**:
The polynomial *approximates* the dataset, meaning it finds a trend minimizing the aggregate squared error, rather than passing directly through every point. As observed, the polynomial handles the massive COVID shocks significantly better than the spline; the spline wildly overshoots out of bounds before and after COVID. The polynomial smoothly sweeps through, barely registering the shock due to the inertia given by the remaining data. 

**2. Least Squares Linear Model Excluding COVID Quarters**

Next, we establish a linear model ($P_1(x)$) but purposely exclude COVID quarters (Indices 41 to 45).


In [ ]:
# Exclude COVID quarters: indices corresponding to early 2020 to early 2021
mask = (x_subset < 41) | (x_subset > 45)
x_clean = x_subset[mask]
y_clean = y_subset[mask]

A_lin = np.vstack([np.ones_like(x_clean), x_clean]).T
coeffs_lin = np.linalg.solve(A_lin.T @ A_lin, A_lin.T @ y_clean)
slope = coeffs_lin[1]

print(f"Calculated Linear Slope: {slope:.4f}% per quarter")

# Plotting the Linear Trend
y_lin_eval = coeffs_lin[0] + coeffs_lin[1] * x_full

plt.figure(figsize=(10, 5))
plt.scatter(x_clean, y_clean, color='green', label='Standard Quarters')
plt.scatter(x_subset[~mask], y_subset[~mask], color='red', label='Excluded COVID Quarters')
plt.plot(x_full, y_lin_eval, 'k-', label='Linear Trend (Excl. COVID)')
plt.title("Least Squares Linear Model (Excluding COVID)")
plt.xlabel("Quarters")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()

# Residual Plot
y_lin_clean_fit = coeffs_lin[0] + coeffs_lin[1] * x_clean
resid_lin = y_clean - y_lin_clean_fit
plt.figure(figsize=(10, 4))
plt.scatter(x_clean, resid_lin, color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.title("Residuals: Linear Model (Cleaned)")
plt.xlabel("Quarters")
plt.ylabel("Residual")
plt.grid(True)
plt.show()



**Economic Validity of Linear Trend**:
The calculated least squares linear slope on the cleaned data shows a mild drift over time. An assumption of a simple linear trend requires that the underlying macro-economic capacity grows symmetrically globally over a 14-year period without cyclical fluctuations. Real economies experience business cycles (recessions, expansions) which are heavily constrained in standard linear models—often making simple linear regression a weak choice for un-transformed economic data, aside from observing the highly abstracted, very long-term macro trend line.


## Part C — Method Justification

**Recommendation for Policymakers**:
For a policymaker interpolating a missing quarter amid relatively stable economic periods, an **Polynomial Fit Approximation** is heavily recommended over an exact Spline Interpolant when dealing with historically volatile data. 
While Cubic Splines are incredibly smooth mathematically ($C^2$ continuity), they are notoriously over-sensitive to strong outliers (like the -28% and +34.9% quarters). Because economic policy relies extensively on historical trends rather than single catastrophic events, the polynomial naturally smooths over outliers to observe the core behavior. 

## Part C Addendum: Big O Analysis

**Big O Notation Theory**: Big O Notation describes the worst-case asymptotic time/space complexity of an algorithm relative to the size of the input, $n$. It bounds an algorithm's growth rate.

- **Natural Cubic Spline Generation**: We mapped a $4(n-1)$ sized system into an $(n-2) 	imes (n-2)$ strictly tridiagonal system. Standard matrix inversion using Gaussian Elimination takes $\mathcal{O}(n^3)$. However, the custom implementation of the **Thomas Algorithm** specifically targets tridiagonal matrices in exactly two sweeps (forward elimination, back substitution), reducing the interpolation cost fundamentally to just $\mathcal{O}(n)$.
- **Polynomial Normal Equations**: Constructing the $m 	imes d$ Vandermonde matrix and computing $A^T A$ requires $\mathcal{O}(m d^2)$ operations. Inversing the resulting $(d+1) 	imes (d+1)$ system via an exact solver implies an additional $\mathcal{O}(d^3)$. Giving a total complexity of $\mathcal{O}(m d^2 + d^3)$. Since here $d=4$ is tightly constrained, the time is effectively bound linearly via $\mathcal{O}(m)$.

**Does Big O impact the recommendation?**
No. Both processes scale efficiently. $n$ (quarterly data points) will functionally never exceed magnitudes ($n \sim 1000$) where polynomial construction or spline tridiagonal solvers bottleneck a modern computer. Sensitivity and mathematical accuracy heavily supersede execution speed for economic datasets.

## Conclusion

This project underscored that data fitting extends well beyond brute mathematics into contextual application. Approximating with a lower-order Least Squares polynomial proved exceptional for extracting long-term macroeconomic trends, gracefully dampening the massive COVID shock. Conversely, Natural Cubic Splines proved superior at faithfully reflecting localized changes, at the catastrophic cost of numerical oscillation near volatile outliers. Splines excel at physically interpolated structures, whereas Least Squares approximations rule statistical trend abstraction.
